In [ ]:
import sqlite3
import sqlite_vec
import ollama
import pandas as pd
from transformers import AutoModel
import torch
from graphqlite import Graph

In [11]:
conn = sqlite3.connect('sqlite.db')
cursor = conn.cursor()
conn.enable_load_extension(True)
sqlite_vec.load(conn)
g = Graph("splite.db")


In [33]:
res = conn.execute("""SELECT name FROM sqlite_schema 
WHERE type='table' 
AND name NOT LIKE 'sqlite_%'
ORDER BY name;
""").fetchall()
res

[('entities',),
 ('episodic_logs',),
 ('vector_fact_mem',),
 ('vector_fact_mem_chunks',),
 ('vector_fact_mem_info',),
 ('vector_fact_mem_rowids',),
 ('vector_fact_mem_vector_chunks00',),
 ('vector_mem',),
 ('vector_mem_chunks',),
 ('vector_mem_info',),
 ('vector_mem_rowids',),
 ('vector_mem_vector_chunks00',)]

In [ ]:
# CREATE_ENT_TABLE_CMD="""
# CREATE TABLE IF NOT EXISTS entities(
#     id INTEGER PRIMARY KEY AUTOINCREMENT, --id matches to id in graphqlite
#     entity_id
#     entity TEXT NOT NULL,
#     last_accessed TIMESTAMP DEFAULT CURRENT_TIMESTAMP
# );
# """

# CREATE_EPISODIC_LOG_CMD = """
# CREATE TABLE IF NOT EXISTS episodic_logs(
#     id INTEGER PRIMARY KEY AUTOINCREMENT,
#     timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
#     role TEXT NOT NULL, -- 'user', 'agent', or 'system_reflection'
#     content TEXT NOT NULL, -- The raw text/conversation turn
#     metadata TEXT -- JSON string for tags, session_ids, etc.
# )

# """

# CREATE_VECTOR_MEM_CMD="""
# CREATE VIRTUAL TABLE IF NOT EXISTS vector_mem using vec0(
#     id INTEGER PRIMARY KEY,
#     embeddings float[256]

# )



# CREATE_VECTOR_FACT_MEM_CMD = """
# CREATE VIRTUAL TABLE IF NOT EXISTS vector_fact_mem using vec0(
#     node_id INTEGER PRIMARY KEY,
#     node_embedding float[256]
# )
# """

In [ ]:
# res = cursor.execute(CREATE_TABLE_CMD)
# res = cursor.execute(CREATE_EPISODIC_LOG_CMD)
res = cursor.execute()
conn.commit()

In [30]:
conn.commit()

In [7]:



model = AutoModel.from_pretrained(
    "jinaai/jina-embeddings-v5-text-nano",
    trust_remote_code=True,
    force_download=True,  # Forces a fresh copy of the code from HF Hub
    dtype=torch.bfloat16
)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device=device)

Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 62953.91it/s]


In [ ]:
query_embedding = model.encode(texts=['Testing'], task = 'retrieval', prompt_name = "document")[0]
len(query_embedding)
query_embedding = query_embedding[:256].tolist()

In [47]:
res = pd.read_sql_query("SELECT * FROM vector_mem", conn)
res

,id,embeddings
0,1,b'\x00\x00\xd0\xbd\x00\x00\xd1\xbc\x00\x00O;\x...
1,2,b'\x00\x00\xd0\xbd\x00\x00\xd1\xbc\x00\x00O;\x...
2,3,b'\x00\x00\xef\xbd\x00\x00\x90\xbc\x00\x00P<\x...


In [ ]:
def store_episodic_log(usr_text:str, agent_respone:str):
    combined = f"User asked [f{usr_text}] agent respond [{agent_respone}]"
    


In [ ]:
import sqlite3
import sqlite_vec
import graphqlite
import torch
import pandas as pd
from transformers import AutoModel
class mem_class:
    conn = None
    cursor = None
    embedding_model = None
    def __init__(self):
        self.conn = sqlite3.connect("/Users/chielerli/Programming/Agent_Instructions/agent_workspace/memory/sqlite.db")
        self.conn.enable_load_extension(True)
        sqlite_vec.load(self.conn)
        self.cursor = self.conn.cursor()
        model = AutoModel.from_pretrained(
            "jinaai/jina-embeddings-v5-text-nano",
            trust_remote_code=True,
            force_download=True,  # Forces a fresh copy of the code from HF Hub
            dtype=torch.bfloat16
        )
        device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
        self.embedding_model = model.to(device=device)

    def get_embeddings(self, input:str, prompt_name="document"):
        emb = self.embedding_model.encode(texts=[input], task = 'retrieval', prompt_name = prompt_name)[0]
        return emb[:256].tolist()
    
    def save_turn(self, role:str, content:str, metadata: None):
        raw_vec = self.get_embeddings(content)
        self.cursor.execute(
            "INSERT INTO episodic_logs (role, content, metadata) VALUES(?, ?, ?)",
            (role, content, metadata)
        )
        log_id =self.cursor.lastrowid
        vector = sqlite_vec.serialize_float32(raw_vec)
        self.cursor.execute(
            "INSERT INTO vector_mem (id, embeddings) VALUES(?, ?)",
            (log_id, vector)

        )


    def search_related_episodic_logs(self, input:str, k = 3):
        raw_vec = self.get_embeddings(input, prompt_name="query")
        vector = sqlite_vec.serialize_float32(raw_vec)
        knn = """select
                v.id,
                e.content
                from vector_mem v
                join episodic_logs e on e.id = v.id
                where embeddings match (?)
                and k = (?)"""
        return pd.read_sql_query(knn, self.conn, params=[vector, k])

    def commit_all(self):
        self.conn.commit()

In [65]:
mem = mem_class()

Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 163680.16it/s]


In [45]:
mem.save_turn("USER", "SECOND SQL INTEGRATION TEST", "INITIAL T")
mem.conn.commit()

In [26]:
from ollama import Client

client = Client(host ="127.0.0.1:11434")


SYS_MSG="""Extract only strict computer science, programming languages, tools, frameworks, and technical software architecture terms from the User Text. 
Extract alongside them their relationship to each other, such as depends_on, uses, is_a, references, calls, part_of
Return the result ONLY as a raw, valid JSON array of lowercase strings. Do not write any introductory text, markdown formatting, or explanations. Only source, target, relationship.
Example: [{"target": Ollama, "source": Python, "relationship": uses}]

TEXT TO ANALYZE
"""
response = client.chat(
    model = "llama3.2:1b",
    messages = [{"role":"user", "content": SYS_MSG+"""The reason your Python script feels slower than the CLI isn't the model itself—it's usually how the data is being handled. By default, the Ollama CLI streams tokens instantly as they are generated, whereas a standard Python API call waits for the entire response to finish generating before returning it to you.

Furthermore, if you are initializing the client or loading the model inside a loop, you introduce massive latency.

To make Python run Ollama at native CLI speeds, you need to use streaming, asynchronous execution, and reuse the client connection.
1. Use the Official ollama Library with Streaming

Streaming is the single biggest factor in perceived speed. Instead of waiting 5 seconds for a full paragraph, tokens will print to your terminal instantly.

First, ensure you have the official library installed:"""}]
)

print(response["message"]["content"])

[
{"target": "Ollama", "source": "Python", "relationship": "depends_on"},
{"target": "python", "source": "Python", "relationship": "uses"},
{"target": "Streaming", "source": "Python", "relationship": "calls"},
{"target": "Ollama Library", "source": "Python", "relationship": "part_of"}
]


In [ ]:
SYS_MSG="""
Extract only strict computer science, programming languages, tools, frameworks, and technical software architecture terms alongside named entities from the User Text. 
Return the result ONLY as a raw, valid array of lowercase strings. Do not write any introductory text, markdown formatting, or explanations. Only source, target, relationship.
Example: ["python", "ollama"]

TEXT TO ANALYZE:
"""

response = client.chat(model="llama3.2:1b",
    messages = [{"role":"user", "content": SYS_MSG+"""The reason your Python script feels slower than the CLI isn't the model itself—it's usually how the data is being handled. By default, the Ollama CLI streams tokens instantly as they are generated, whereas a standard Python API call waits for the entire response to finish generating before returning it to you.

Furthermore, if you are initializing the client or loading the model inside a loop, you introduce massive latency.

To make Python run Ollama at native CLI speeds, you need to use streaming, asynchronous execution, and reuse the client connection.
1. Use the Official ollama Library with Streaming

Streaming is the single biggest factor in perceived speed. Instead of waiting 5 seconds for a full paragraph, tokens will print to your terminal instantly.

First, ensure you have the official library installed:"""}])
print(response)

model='llama3.2:1b' created_at='2026-07-07T01:21:36.369161Z' done=True done_reason='stop' total_duration=1133319166 load_duration=688657625 prompt_eval_count=264 prompt_eval_duration=216705000 eval_count=17 eval_duration=226151000 message=Message(role='assistant', content='["ollama", "python", "streaming", "asynchronous execution"]', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None
